# Pythia-410m: GaugeMLA (our method) only, initialisation and 50M-token training

GaugeMLA exclusively: no baseline is built or trained. The full experiments are in the other three notebooks.

Requires **Google Colab Pro+**. **Runtime: A100 GPU, High-RAM** (Runtime → Change runtime type); the initialisation runs on its CPU.

Builds GaugeMLA's t = 0 checkpoints at 25, 12.5 and 6.25% of the KV cache into `MyDrive/gaugemla/grad_clip/p410m_gc/init/ckpt/`, then distils each toward Pythia-2.8b for 50M tokens with the gradient clipped at 1.0, saving each final model under `MyDrive/gaugemla/grad_clip/p410m_gc/runs/<stamp>/models/`. `TEST_TOKENS = 50000` is a quick test under its own stamp; 0 is the full run. Finished checkpoints and branches are skipped on a rerun.

Requires a Hugging Face token: add it as the Colab secret `HF_TOKEN` and give this notebook access.

In [ ]:
# @title Setup
TEST_TOKENS = 0  # @param {type:"integer"}

import os, subprocess, sys
from google.colab import drive
drive.mount("/content/drive")
# The repository is public. While it is private, uncomment these lines and add the Colab secret READ_MLA_TOKEN
# (a GitHub token that can read it):
# from google.colab import userdata
# os.environ["READ_MLA_TOKEN"] = userdata.get("READ_MLA_TOKEN")
# with open("/content/askpass.sh", "w") as f:
#     f.write('#!/bin/sh\necho "$READ_MLA_TOKEN"\n')
# os.chmod("/content/askpass.sh", 0o700)
# os.environ["GIT_ASKPASS"] = "/content/askpass.sh"
os.environ["GIT_TERMINAL_PROMPT"] = "0"
if os.path.isdir("/content/repo/.git"):
    subprocess.run(["git", "-C", "/content/repo", "fetch", "--depth", "1", "origin", "main"], check=True)
    subprocess.run(["git", "-C", "/content/repo", "reset", "--hard", "origin/main"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
                    "https://github.com/anon5159753/gaugemla.git", "/content/repo"], check=True)
    subprocess.run(["git", "-C", "/content/repo", "sparse-checkout", "set", "src", "colab"], check=True)
sys.path.insert(0, "/content/repo/colab")
import launch

In [ ]:
# @title GaugeMLA t = 0 checkpoints
launch.run("p410m_gc", "init", DRY_RUN=False, INIT_ARMS="mla_w64_ours_unrope_cost")

In [ ]:
# @title GaugeMLA training, 50M tokens
for branch in ('mla_w64_ours_unrope_cost_G4', 'mla_w64_ours_unrope_cost_G2', 'mla_w64_ours_unrope_cost_G1'):
    launch.run("p410m_gc", "run", BRANCH=branch, TEST_TOKENS=TEST_TOKENS)

In [ ]:
# @title KL to the teacher at t = 0 and at the end of training, beside results/
launch.kl_table("p410m_gc", TEST_TOKENS)